# Egyptian National ID — Data Preparation & EDA

End-to-end notebook that:
1. Validates dataset structure and annotations
2. Visualizes class distributions, bounding-box statistics, and image quality
3. Demonstrates the preprocessing pipeline (resize, enhance, deskew)
4. Surfaces any data issues before training

**Dataset:** Roboflow — Egyptian Person ID v1  
**Classes (15):** Add1, Add2, Back, ExpDate, First_Name, Front, Gender, HusbandName, ID, IssueDate, Job, Last_Name, Religion, Serial_Num, Status

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# ── project root on sys.path so src.* imports work ──────────────────────────
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET_ROOT = ROOT / "Egyptain-Person-ID-1"
print(f"Project root : {ROOT}")
print(f"Dataset root : {DATASET_ROOT}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, random, textwrap
import numpy as np
import pandas as pd
import yaml
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from src.data.validate import DatasetValidator
from src.data.preprocess import (
    load_image, resize_with_padding, enhance_for_ocr, deskew
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
print("Imports OK")

## 2. Constants

In [ ]:
CLASSES = [
    "Add1", "Add2", "Back", "ExpDate", "First_Name", "Front",
    "Gender", "HusbandName", "ID", "IssueDate", "Job",
    "Last_Name", "Religion", "Serial_Num", "Status"
]
IDX2CLS = {i: c for i, c in enumerate(CLASSES)}

FRONT_FIELDS = {"First_Name", "Last_Name", "ID", "Gender", "Religion", "HusbandName", "Job", "Front"}
BACK_FIELDS  = {"Add1", "Add2", "Serial_Num", "IssueDate", "ExpDate", "Status", "Back"}

# palette — one colour per class
PALETTE = plt.cm.get_cmap("tab20", len(CLASSES))
CLS_COLORS = {cls: PALETTE(i)[:3] for i, cls in enumerate(CLASSES)}

# which splits actually exist on disk
SPLITS = [s for s in ["train", "valid", "test"] if (DATASET_ROOT / s / "images").exists()]
print("Available splits:", SPLITS)

## 3. Dataset Validation

In [ ]:
validator = DatasetValidator(str(DATASET_ROOT))
ok = validator.run()
report = validator.report()

status_icon = "✅" if ok else "⚠️"
print(f"{status_icon}  Dataset valid: {ok}")

if report["issues"]:
    print("\nIssues found:")
    for issue in report["issues"]:
        print(f"  • {issue}")

print("\nSplit counts from validator:")
for split, counts in report["stats"].get("split_counts", {}).items():
    print(f"  {split:8s}: {counts['images']:4d} images  {counts['labels']:4d} labels")

## 4. Split Statistics

In [ ]:
def parse_label_file(label_path: Path):
    """Parse a YOLO label file; handles both bbox (5 vals) and polygon (>5 vals)."""
    annotations = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            if len(coords) == 4:
                # standard YOLO: cx cy w h
                cx, cy, w, h = coords
                fmt = "bbox"
            elif len(coords) >= 8:
                # polygon / OBB: interleaved x1 y1 x2 y2 ...
                xs = coords[0::2]
                ys = coords[1::2]
                cx = (min(xs) + max(xs)) / 2
                cy = (min(ys) + max(ys)) / 2
                w  = max(xs) - min(xs)
                h  = max(ys) - min(ys)
                fmt = "polygon"
            else:
                continue  # malformed
            annotations.append({
                "class_id": cls_id,
                "class_name": IDX2CLS.get(cls_id, f"unk_{cls_id}"),
                "cx": cx, "cy": cy, "w": w, "h": h,
                "area": w * h,
                "aspect_ratio": w / h if h > 0 else 0,
                "format": fmt,
            })
    return annotations


def collect_split_data(split: str):
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    records = []
    for img_path in sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png")):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h_px, w_px = img.shape[:2]
        anns = parse_label_file(lbl_path) if lbl_path.exists() else []
        for ann in anns:
            records.append({
                "split": split,
                "image": img_path.name,
                "img_w": w_px, "img_h": h_px,
                "file_kb": img_path.stat().st_size / 1024,
                **ann,
            })
        if not anns:
            records.append({
                "split": split, "image": img_path.name,
                "img_w": w_px, "img_h": h_px,
                "file_kb": img_path.stat().st_size / 1024,
                "class_id": None, "class_name": None,
                "cx": None, "cy": None, "w": None, "h": None,
                "area": None, "aspect_ratio": None, "format": None,
            })
    return records


print("Collecting annotation data (may take ~30 s for large splits)...")
all_records = []
for split in SPLITS:
    recs = collect_split_data(split)
    all_records.extend(recs)
    print(f"  {split}: {len(recs)} annotation rows")

df = pd.DataFrame(all_records)
ann_df = df.dropna(subset=["class_name"]).copy()
print(f"\nTotal annotations: {len(ann_df)}")

In [ ]:
# per-split image & annotation summary
summary = (
    df.groupby("split")
    .agg(
        unique_images=("image", "nunique"),
        total_annotations=("class_id", "count"),
    )
    .reset_index()
)
summary["ann_per_image"] = (summary["total_annotations"] / summary["unique_images"]).round(2)
display(summary)

In [ ]:
# annotation format breakdown
fmt_counts = ann_df.groupby(["split", "format"]).size().unstack(fill_value=0)
print("Annotation format breakdown:")
display(fmt_counts)

## 5. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, len(SPLITS), figsize=(7 * len(SPLITS), 5), sharey=False)
if len(SPLITS) == 1:
    axes = [axes]

for ax, split in zip(axes, SPLITS):
    subset = ann_df[ann_df["split"] == split]
    counts = subset["class_name"].value_counts().reindex(CLASSES, fill_value=0)
    colors = [CLS_COLORS[c] for c in counts.index]
    bars = ax.barh(counts.index[::-1], counts.values[::-1], color=colors[::-1], edgecolor="white")
    ax.set_title(f"{split}  ({subset['image'].nunique()} images)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Annotation count")
    for bar, val in zip(bars, counts.values[::-1]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(val), va="center", fontsize=9)

fig.suptitle("Class Distribution per Split", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# class imbalance ratio (max/min annotation count in train)
if "train" in SPLITS:
    train_counts = ann_df[ann_df["split"] == "train"]["class_name"].value_counts()
    ratio = train_counts.max() / train_counts.min()
    print(f"Train imbalance ratio (max/min): {ratio:.1f}x")
    print(f"  Most common : {train_counts.idxmax()} ({train_counts.max()})")
    print(f"  Least common: {train_counts.idxmin()} ({train_counts.min()})")

## 6. Image Quality Analysis

In [ ]:
img_df = df.drop_duplicates(subset=["split", "image"]).copy()
img_df["aspect_ratio_img"] = img_df["img_w"] / img_df["img_h"]
img_df["resolution"] = img_df["img_w"].astype(str) + "x" + img_df["img_h"].astype(str)

print("Image dimension summary:")
display(
    img_df.groupby("split")[["img_w", "img_h", "file_kb", "aspect_ratio_img"]]
    .describe().round(1)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# width distribution
for split in SPLITS:
    subset = img_df[img_df["split"] == split]
    axes[0].hist(subset["img_w"], bins=20, alpha=0.7, label=split)
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width (px)")
axes[0].legend()

# height distribution
for split in SPLITS:
    subset = img_df[img_df["split"] == split]
    axes[1].hist(subset["img_h"], bins=20, alpha=0.7, label=split)
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height (px)")
axes[1].legend()

# file size distribution
for split in SPLITS:
    subset = img_df[img_df["split"] == split]
    axes[2].hist(subset["file_kb"], bins=20, alpha=0.7, label=split)
axes[2].set_title("File Size Distribution")
axes[2].set_xlabel("Size (KB)")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# flag non-square / unusual aspect ratios
unusual = img_df[(img_df["aspect_ratio_img"] < 0.5) | (img_df["aspect_ratio_img"] > 2.5)]
print(f"Images with unusual aspect ratio (<0.5 or >2.5): {len(unusual)}")
if len(unusual):
    display(unusual[["split", "image", "img_w", "img_h", "aspect_ratio_img"]].head(10))

## 7. Bounding Box Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train_ann = ann_df[ann_df["split"] == SPLITS[0]]  # use first available split

# box width
axes[0].hist(train_ann["w"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title(f"Box Width (normalised) — {SPLITS[0]}")
axes[0].set_xlabel("Width")

# box height
axes[1].hist(train_ann["h"], bins=40, color="coral", edgecolor="white")
axes[1].set_title(f"Box Height (normalised) — {SPLITS[0]}")
axes[1].set_xlabel("Height")

# box area
axes[2].hist(train_ann["area"], bins=40, color="mediumseagreen", edgecolor="white")
axes[2].set_title(f"Box Area (normalised) — {SPLITS[0]}")
axes[2].set_xlabel("Area")

plt.tight_layout()
plt.show()

In [ ]:
# per-class box area (violin plot)
fig, ax = plt.subplots(figsize=(14, 5))
order = ann_df.groupby("class_name")["area"].median().sort_values(ascending=False).index
sns.violinplot(
    data=ann_df, x="class_name", y="area",
    order=order, palette="muted", inner="quartile", ax=ax
)
ax.set_title("Bounding Box Area Distribution per Class", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Normalised Area")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

## 8. Spatial Distribution Heatmaps

Where do fields tend to appear on the card? Bright = many annotations in that region.

In [ ]:
HEAT_CLASSES = ["First_Name", "Last_Name", "ID", "Gender", "Add1", "Add2"]
heat_subset = [c for c in HEAT_CLASSES if c in ann_df["class_name"].unique()]

cols = 3
rows = (len(heat_subset) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).flatten()

cmap_hot = LinearSegmentedColormap.from_list("heat", ["#1a1a2e", "#e94560", "#f5a623"])

for ax, cls_name in zip(axes, heat_subset):
    sub = ann_df[ann_df["class_name"] == cls_name]
    heatmap, _, _ = np.histogram2d(
        sub["cx"].dropna(), sub["cy"].dropna(),
        bins=32, range=[[0, 1], [0, 1]]
    )
    ax.imshow(heatmap.T, origin="lower", extent=[0, 1, 0, 1],
              cmap=cmap_hot, interpolation="bilinear")
    ax.set_title(cls_name, fontweight="bold")
    ax.set_xlabel("x (left → right)")
    ax.set_ylabel("y (bottom → top)")

for ax in axes[len(heat_subset):]:
    ax.set_visible(False)

fig.suptitle("Field Centre Heatmaps", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Sample Images with Annotations

In [ ]:
def draw_annotations(img_path: Path, lbl_path: Path) -> np.ndarray:
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if not lbl_path.exists():
        return img
    for ann in parse_label_file(lbl_path):
        cx, cy, bw, bh = ann["cx"], ann["cy"], ann["w"], ann["h"]
        x1 = int((cx - bw / 2) * w)
        y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w)
        y2 = int((cy + bh / 2) * h)
        color = tuple(int(c * 255) for c in CLS_COLORS[ann["class_name"]])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = ann["class_name"]
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(img, (x1, y1 - th - 4), (x1 + tw + 4, y1), color, -1)
        cv2.putText(img, label, (x1 + 2, y1 - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    return img


def show_samples(split: str, n: int = 6, seed: int = 42):
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    paths = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    random.seed(seed)
    sample = random.sample(paths, min(n, len(paths)))

    cols = 3
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, sample):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        annotated = draw_annotations(img_path, lbl_path)
        ax.imshow(annotated)
        ax.set_title(textwrap.shorten(img_path.name, 40), fontsize=8)
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.set_visible(False)

    fig.suptitle(f"Sample Annotated Images — {split}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


for split in SPLITS:
    show_samples(split, n=6)

## 10. Preprocessing Pipeline Demo

Shows each preprocessing step on a sample image so you can verify outputs before training.

In [ ]:
def demo_preprocessing(img_path: Path):
    original = load_image(str(img_path))
    resized, scale, pad = resize_with_padding(original, target_size=640)
    enhanced = enhance_for_ocr(original)
    deskewed = deskew(original)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    steps = [
        (original,  f"Original\n{original.shape[1]}×{original.shape[0]} px"),
        (resized,   f"Letterbox Resize → 640×640\nscale={scale:.3f}, pad={pad}"),
        (enhanced,  "CLAHE + Denoise\n(enhance_for_ocr)"),
        (deskewed,  "Deskew\n(deskew)"),
    ]
    for ax, (img, title) in zip(axes, steps):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    fig.suptitle(f"Preprocessing Demo — {img_path.name}", fontweight="bold")
    plt.tight_layout()
    plt.show()


# pick one image from each available split
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    img_paths = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    if img_paths:
        demo_preprocessing(img_paths[0])

## 11. OCR Field Crop Demo

Crop individual detected fields and show the CLAHE-enhanced version that EasyOCR will receive.

In [ ]:
from src.data.preprocess import crop_field

def demo_field_crops(split: str, n_images: int = 2, seed: int = 7):
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    paths = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    random.seed(seed)
    sample = random.sample(paths, min(n_images, len(paths)))

    for img_path in sample:
        img = load_image(str(img_path))
        h, w = img.shape[:2]
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        anns = parse_label_file(lbl_path) if lbl_path.exists() else []
        if not anns:
            continue

        n_crops = min(len(anns), 8)
        fig, axes = plt.subplots(2, n_crops, figsize=(3 * n_crops, 5))
        if n_crops == 1:
            axes = axes.reshape(2, 1)

        for col, ann in enumerate(anns[:n_crops]):
            cx, cy, bw, bh = ann["cx"], ann["cy"], ann["w"], ann["h"]
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            x2 = (cx + bw / 2) * w
            y2 = (cy + bh / 2) * h
            crop = crop_field(img, (x1, y1, x2, y2), padding=4)
            if crop.size == 0:
                continue
            enhanced = enhance_for_ocr(crop)

            axes[0, col].imshow(crop)
            axes[0, col].set_title(ann["class_name"], fontsize=8, fontweight="bold")
            axes[0, col].axis("off")

            axes[1, col].imshow(enhanced)
            axes[1, col].set_title("enhanced", fontsize=7, color="gray")
            axes[1, col].axis("off")

        fig.suptitle(f"Field Crops — {img_path.name}", fontweight="bold")
        plt.tight_layout()
        plt.show()


demo_field_crops(SPLITS[0], n_images=2)

## 12. Missing Label Check

In [ ]:
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    images = {p.stem for p in img_dir.glob("*.jpg")} | {p.stem for p in img_dir.glob("*.png")}
    labels = {p.stem for p in lbl_dir.glob("*.txt")}
    unlabeled = images - labels
    extra_labels = labels - images
    print(f"[{split}] images={len(images)}, labels={len(labels)}, "
          f"unlabeled={len(unlabeled)}, orphan_labels={len(extra_labels)}")
    if unlabeled:
        print(f"  Unlabeled sample: {list(unlabeled)[:3]}")

## 13. Duplicate Image Check

In [ ]:
import hashlib

def file_md5(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

print("Computing image hashes (may take a moment)...")
hash_map: dict = {}
dupes = []
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    for p in sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png")):
        h = file_md5(p)
        if h in hash_map:
            dupes.append((hash_map[h], f"{split}/{p.name}"))
        else:
            hash_map[h] = f"{split}/{p.name}"

print(f"Total unique images: {len(hash_map)}")
print(f"Exact duplicates found: {len(dupes)}")
if dupes:
    for a, b in dupes[:5]:
        print(f"  {a}  ==  {b}")

## 14. Summary & Recommendations

In [ ]:
print("=" * 60)
print("DATA PREPARATION SUMMARY")
print("=" * 60)

for _, row in summary.iterrows():
    print(f"  {row['split']:8s}: {row['unique_images']:4d} images  "
          f"{row['total_annotations']:5d} annotations  "
          f"({row['ann_per_image']} ann/img)")

print()

# validation status
if report["issues"]:
    print(f"⚠️  {len(report['issues'])} validation issue(s):")
    for issue in report["issues"]:
        print(f"   • {issue}")
else:
    print("✅ Dataset passed all validation checks")

# class imbalance
if "train" in SPLITS:
    tc = ann_df[ann_df["split"] == "train"]["class_name"].value_counts()
    imbalance = tc.max() / tc.min()
    flag = "⚠️" if imbalance > 5 else "✅"
    print(f"{flag}  Class imbalance (train): {imbalance:.1f}x")

# duplicates
flag = "⚠️" if dupes else "✅"
print(f"{flag}  Duplicate images: {len(dupes)}")

# missing valid split
if "valid" not in SPLITS:
    print("⚠️  No 'valid' split found — YOLO will use test set for validation")
    print("   Consider creating a validation split from train data")

print()
print("Next steps: run `make train` or `dvc repro` to start training.")
print("=" * 60)